# 06 · Smoke-test training diagnosis (read-only by default)

Background: in the notebook-05 smoke test the gradient norm became exactly 0 after epoch 1, the training loss
was identical in epochs 2 and 3, and every validation correlation was exactly 0. The local investigation
(`docs/SMOKE_TRAINING_DIAGNOSIS.md`) attributes this to **prediction collapse**: every raw output of the
official prediction heads falls below their `clamp(min=0)`, after which the gradient is exactly zero.

* **Part A, read-only, no training (≈ 5–10 min, GPU optional).** Loads the existing smoke checkpoint,
  reports the raw head outputs before the clamp, and compares the logged training loss with the loss of
  the all-zero predictor on all training graphs. Nothing in `runs/` is modified.
* **Part B, short GPU diagnostic, OFF by default (≈ 5–15 min on the A100).** 60 production training steps
  on 6 training graphs in FP32 and in FP16, instrumented step by step. Set `RUN_GPU_DIAGNOSTIC = True`
  only after you have authorised it. It writes only `reports/06_*.json`.

No full training and no precision benchmark is started from this notebook. Notebook 02 is not needed
(only processed data on Drive is read).

In [ ]:
# ── Bootstrap (identical in every notebook; safe to re-run) ──────────────────────
# Mounts YOUR Google Drive (you authorise in the popup; no credentials are stored),
# unpacks the project code from Drive and puts it on the Python path.
import os, sys, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/VeriHGN'            # persistent project storage
CODE_ZIP   = f'{DRIVE_ROOT}/code/verihgn_code.zip'        # uploaded from the laptop (tools/make_colab_bundle.py)
CODE_DIR   = '/content/verihgn-project'

if not os.path.exists(CODE_ZIP):
    raise FileNotFoundError(f'Upload verihgn_code.zip to {CODE_ZIP} first (see README, "Colab").')
stamp = f'{CODE_DIR}/.zip_mtime'
if not os.path.exists(stamp) or open(stamp).read() != str(os.path.getmtime(CODE_ZIP)):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(CODE_DIR)
    open(stamp, 'w').write(str(os.path.getmtime(CODE_ZIP)))
sys.path.insert(0, f'{CODE_DIR}/src')

marker = '/content/.verihgn_deps_ok'
if not os.path.exists(marker):
    from verihgn_repro.colab import install_dependencies
    install_dependencies()
    open(marker, 'w').write('ok')

from verihgn_repro.colab import DriveLayout
L = DriveLayout().ensure()
print('code:', CODE_DIR, '| drive root:', L.root)

In [ ]:
import json, subprocess, sys, time
from pathlib import Path
from verihgn_repro.config import load_config
SCOPE = 'official9'
CFG_DIR = Path(CODE_DIR) / 'configs'
PHASH = json.load(open(L.reports / f'03_preprocessing_{SCOPE}.json'))['prep_hash']
PROCESSED = L.processed_for(PHASH)
cfg = load_config(CFG_DIR / 'base.yaml')
SPLIT = L.splits / f'{SCOPE}__{cfg.experiment.split}.json'
scfg = load_config(CFG_DIR / 'base.yaml', CFG_DIR / 'smoke.yaml'); scfg.experiment.name += '_smoke'
smoke_dirs = sorted(p for p in L.runs.glob(scfg.run_id() + '*') if (p / 'checkpoints' / 'last.pt').exists())
print('processed:', PROCESSED, '\nsplit:', SPLIT, '\nsmoke runs with a checkpoint:', *smoke_dirs, sep='\n  ')

def new_report(name):
    """Never overwrite an existing report: add a timestamp if the name is taken."""
    p = L.reports / name
    return p if not p.exists() else p.with_name(f'{p.stem}__{time.strftime("%Y%m%d_%H%M%S")}{p.suffix}')

def tool(script, *args):
    r = subprocess.run([sys.executable, f'tools/{script}', *map(str, args)], cwd=CODE_DIR, capture_output=True, text=True)
    print(r.stdout[-6000:], r.stderr[-3000:])
    assert r.returncode == 0, f'{script} failed'

## Part A · Read-only inspection of the existing smoke checkpoint(s)

In [ ]:
for run_dir in smoke_dirs:
    out = new_report(f'06_collapse_inspection__{run_dir.name}.json')
    tool('inspect_collapse.py', '--run-dir', run_dir, '--processed', PROCESSED, '--split', SPLIT, '--out', out)

## Part B · Short instrumented FP32 vs FP16 diagnostic (only when authorised)

In [ ]:
RUN_GPU_DIAGNOSTIC = False      # set True only after authorising this short GPU job
if RUN_GPU_DIAGNOSTIC:
    out = new_report('06_step_diagnostic.json')
    tool('diagnose_training.py', '--processed', PROCESSED, '--split', SPLIT, '--out', out,
         '--precisions', 'fp32', 'fp16', '--steps', 60, '--train-samples', 6, '--val-samples', 2,
         '--metric-every', 20, '--device', 'cuda')
else:
    print('Part B skipped (RUN_GPU_DIAGNOSTIC = False)')
print('→ Send back reports/06_*.json')